# RAG clássico na prática
## Do manual da equipe a uma resposta com fonte

**DCA0305 · Machine Learning-Based Systems Design · UFRN**

> *"What tyre pressure do we use for car 27 in the wet?"*

Essa pergunta atravessa o notebook inteiro. Na aula 7 o modelo respondeu a ela com números inventados, porque o carro 27 só existe nos nossos arquivos. Aqui a mesma pergunta vai passar por um manual de equipe de 12 documentos, ser fatiada, transformada em vetor, guardada num banco, recuperada, colada num prompt e, no fim, respondida com a fonte entre colchetes. Se você conseguir seguir essa pergunta pelo pipeline inteiro, consegue montar um RAG para qualquer conjunto de documentos.

O corpus é o **Aurora Racing Team Handbook**, um manual fictício da equipe da aula 7 (carros 27 e 88). Ele foi escrito para esta aula, então nenhum modelo de linguagem o viu no treinamento. Isso é proposital. Toda resposta certa que aparecer aqui veio da recuperação, não da memória do modelo.

O que este notebook produz no fim é um arquivo `results_baseline.csv` com 20 perguntas, a resposta com e sem RAG, o acerto e o tempo. **Guarde esse arquivo.** Na aula 9 cada variante sofisticada do RAG roda no mesmo corpus, com as mesmas perguntas, e a comparação é contra o que você gerou aqui.

## 📐 Como estudar este notebook

O ritmo é o mesmo da aula 7. Todo bloco tem as mesmas partes, na mesma ordem, com os mesmos ícones.

| Parte | O que é | O que você faz |
|---|---|---|
| 🎯 **Uma ideia** | Uma frase. A única coisa nova do bloco. | Leia. |
| 🔮 **Preveja** | Uma pergunta cuja resposta a próxima célula revela. | Escreva a sua aposta **antes** de rodar. |
| ▶️ **Rode** | Uma célula completa, seguida de 🔍 **O que você deve ver**. | Rode, leia a saída com calma, compare com a aposta. |
| 🧩 **Preencha a lacuna** | Uma célula com `# ---- SEU CÓDIGO AQUI ----`, com 🧪 **Como saber se deu certo** e uma 🔑 **solução de referência** escondida. | Complete antes de seguir. Abra a solução só depois de dez minutos tentando. |
| 🔒 **Fechamento** | Uma frase que você deve conseguir dizer em voz alta. | Diga. |
| 🧭 **Por que o experimento é assim** e 🐇 **Toca do coelho** | Uma nota sobre o desenho do bloco e uma trilha opcional. | Leia a nota. Entre na toca se tiver tempo. |

Fechando cada bloco há um 📓 **Diário de bordo** com duas linhas para preencher.

### O contrato

1. **Aposte antes de rodar.** Errar a previsão e corrigir é o que grava o conteúdo.
2. **A regra dos dez minutos** para as soluções de referência.
3. **Quebre as coisas.** Mude `chunk_size`, mude `top_k`, troque o modelo, apague a regra de grounding. Nada aqui quebra de verdade.
4. **Duas sessões.** Blocos 0 a 2 num dia, Blocos 3 a 5 em outro. O índice do Chroma fica em disco, então a segunda sessão não paga o embedding de novo (no Colab a máquina é apagada e o índice é reconstruído em menos de um minuto).
5. **Escreva no diário.**

Tempo estimado. Cerca de 75 minutos por sessão.

### A interface que a aula 9 vai reaproveitar

Tudo no notebook passa por quatro funções com contrato fixo. `make_chunks(docs, ...)` devolve linhas com texto e metadados. `build_index(rows, name)` guarda. `retrieve(col, question, k)` devolve uma lista de `{text, doc, section, distance}`. `build_prompt(question, hits, grounded)` monta as mensagens. Na aula 9 cada variante (memória, CRAG, HyDE, híbrido...) é uma nova função `retrieve` ou um novo `build_prompt` que respeita o mesmo contrato e roda no mesmo harness do Bloco 5.

## 🧰 Preparação (no próprio Colab)

Igual à aula 7: no Colab o Ollama não vem instalado e a máquina é apagada ao fim da sessão, então a preparação é feita por células e repetida a cada sessão. Escolha **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**. Em CPU tudo funciona, só mais devagar (o gerador de 3B leva uns 20 s por resposta; o modelo de embedding é rápido nos dois casos).

| Passo | O que faz | Quanto tempo |
|---|---|---|
| 1 · Instalar | Instala o Ollama na máquina virtual. | ~1 min |
| 2 · Subir | Sobe o servidor em segundo plano. | segundos |
| 3 · Baixar | Baixa o gerador e o modelo de embedding. | ~2 min |
| 4 · Corpus | Baixa os 12 documentos do manual e as 20 perguntas. | segundos |

Se você estiver rodando na sua máquina com o Ollama já instalado, pule os passos 1 a 3 e rode só o 4 e a instalação dos pacotes.

In [ ]:
# Passo 1 · Instala o Ollama na máquina virtual do Colab.
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 3
!ollama --version

In [ ]:
# Passo 2 · Sobe o servidor em segundo plano e espera até ele responder.
import subprocess, time, requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)
print("servidor no ar:", server_is_up(), requests.get(f"{OLLAMA_URL}/api/version").json())

In [ ]:
# Passo 3 · Baixa os modelos. O gerador é o mesmo da aula 7; o modelo de embedding é novo.
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:0.5b
!ollama pull nomic-embed-text
!ollama list

In [ ]:
# Passo 4 · Corpus e perguntas. Baixa do repositório do curso se a pasta local não existir.
import os, requests
RAW = "https://raw.githubusercontent.com/ivanovitchm/aiengineering/main/lessons/week06"
os.makedirs("corpus", exist_ok=True)
for f in ['car_specs.md', 'comms_protocol.md', 'drivers.md', 'fuel_and_energy.md', 'garage_and_pitlane.md', 'handbook_procedures.md', 'handbook_tyres.md', 'logistics.md', 'procedure_codes.md', 'radio_glossary.md', 'sponsor_obligations.md', 'strategy_playbook.md']:
    p = os.path.join("corpus", f)
    if not os.path.exists(p):
        open(p, "w", encoding="utf-8").write(requests.get(f"{RAW}/corpus/{f}").text)
if not os.path.exists("questions.json"):
    open("questions.json", "w", encoding="utf-8").write(requests.get(f"{RAW}/questions.json").text)
print(sorted(os.listdir("corpus")))

In [ ]:
# Instala o que falta. Rode uma vez por ambiente.
!pip -q install openai chromadb requests pandas matplotlib scikit-learn

In [ ]:
# Núcleo do RAG clássico. As mesmas funções servem à demo em sala e ao notebook completo.
import os, re, glob, json, time, requests
from openai import OpenAI
import chromadb

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "qwen2.5:3b"            # o gerador (o Léo). Troque por "qwen2.5:0.5b" se a máquina sofrer.
EMB_MODEL  = "nomic-embed-text"      # o modelo de embedding. Tem de ser O MESMO na indexação e na consulta.
CORPUS_DIR = "corpus"
CHROMA_DIR = "chroma_lesson08"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

# ---- tokens. Aqui 1 token = 1 palavra (com o espaço que a precede), para o notebook não depender de nada.
# Tokenizadores reais (BPE) produzem cerca de 30% mais tokens que palavras. A proporção entre os tamanhos é o que importa.
def tokenize(text):   return re.findall(r"\s*\S+", text)
def detokenize(toks): return "".join(toks).strip()
def count_tokens(text): return len(tokenize(text))

# ---- Load
def load_corpus(folder=CORPUS_DIR):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*.md"))):
        docs.append({"doc": os.path.basename(path), "text": open(path, encoding="utf-8").read()})
    return docs

# ---- Chunk (tamanho fixo com overlap, em tokens)
def chunk_fixed(text, chunk_size=512, overlap=51):
    toks, step, out = tokenize(text), chunk_size - overlap, []
    for start in range(0, max(len(toks), 1), step):
        out.append(detokenize(toks[start:start + chunk_size]))
        if start + chunk_size >= len(toks):
            break
    return out

def make_chunks(docs, chunk_size=512, overlap=51):
    """Uma linha por chunk: id, texto e metadados (documento, posição, última seção vista)."""
    rows = []
    for d in docs:
        carried = "-"                                          # última seção vista no chunk anterior
        for i, c in enumerate(chunk_fixed(d["text"], chunk_size, overlap)):
            heads = [(m.start(), m.group(1)[:80]) for m in re.finditer(r"^#+ (.+)$", c, re.M)]
            first_half = [h for pos, h in heads if pos < len(c) / 2]
            section = first_half[-1] if first_half else carried   # a seção que domina o chunk
            if heads:
                carried = heads[-1][1]
            rows.append({"id": f"{d['doc']}#{i}", "text": c,
                         "doc": d["doc"], "chunk": i, "section": section})
    return rows

# ---- Embed (Ollama /api/embed devolve uma lista de vetores)
def embed(texts, model=EMB_MODEL):
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": texts})
    r.raise_for_status()
    return r.json()["embeddings"]

# ---- Store (Chroma persistente, distância = 1 - cosseno)
def build_index(rows, name, persist_dir=CHROMA_DIR, batch=32, rebuild=False):
    store = chromadb.PersistentClient(path=persist_dir)
    if rebuild:
        try: store.delete_collection(name)
        except Exception: pass
    col = store.get_or_create_collection(name, metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL})
    if col.count() == len(rows):
        return col                                   # já indexado; não paga o embedding de novo
    for i in range(0, len(rows), batch):
        part = rows[i:i + batch]
        col.upsert(ids=[r["id"] for r in part], documents=[r["text"] for r in part],
                   embeddings=embed([r["text"] for r in part]),
                   metadatas=[{"doc": r["doc"], "chunk": r["chunk"], "section": r["section"]} for r in part])
    return col

# ---- Retrieve
def retrieve(col, question, k=4):
    assert col.metadata.get("embedding_model") == EMB_MODEL, "índice e consulta com modelos de embedding diferentes"
    res = col.query(query_embeddings=embed([question]), n_results=k, include=["documents", "metadatas", "distances"])
    return [{"text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

# ---- Augment
GROUNDING = ("You are the pit-wall assistant of Aurora Racing. Answer using ONLY the context below. "
             "If the context does not contain the answer, say exactly: \"The handbook does not cover this.\" "
             "Cite the source of each fact in square brackets, like [handbook_tyres.md].")
FREE = "You are the pit-wall assistant of Aurora Racing. Answer the question."

def build_prompt(question, hits, grounded=True):
    context = "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)
    system = GROUNDING if grounded else FREE
    user = f"Context:\n{context}\n\nQuestion: {question}" if hits else f"Question: {question}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

# ---- Generate
def generate(messages, model=GEN_MODEL, temperature=0.0):
    t0 = time.time()
    resp = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    return resp.choices[0].message.content.strip(), time.time() - t0

def show(hits):
    for i, h in enumerate(hits, 1):
        print(f"[{i}] {h['doc']} · {h['section']}  (distance {h['distance']:.3f})")
        print("    " + h["text"][:160].replace("\n", " ") + " ...")


In [ ]:
# Cronômetro automático. A partir daqui toda célula imprime o seu tempo de execução (⏱) e o registra em CELL_TIMES.
import time
from IPython import get_ipython

CELL_TIMES = []
_ip = get_ipython()

def _pre_run(info):
    _ip._t0 = time.perf_counter()

def _post_run(result):
    dt = time.perf_counter() - getattr(_ip, "_t0", time.perf_counter())
    first_line = (result.info.raw_cell.strip().splitlines() or [""])[0][:70]
    CELL_TIMES.append({"cell": first_line, "seconds": round(dt, 2)})
    print(f"⏱ {dt:.1f} s")

if not getattr(_ip, "_timer_installed", False):          # evita registrar duas vezes se a célula for rerodada
    _ip._t0 = time.perf_counter()
    _ip.events.register("pre_run_cell", _pre_run)
    _ip.events.register("post_run_cell", _post_run)
    _ip._timer_installed = True
print("cronômetro ligado")


---
# Bloco 0 · O corpus e as perguntas

### 🎯 Uma ideia
**Um RAG é avaliado antes de ser construído: corpus fixo, perguntas fixas, respostas de referência fixas. Sem isso, "melhorou" é opinião.**

O manual tem 12 documentos em Markdown, cada um uma seção do handbook da Aurora Racing. As 20 perguntas vêm em três faixas. Oito são **diretas**, a resposta está numa frase de um documento. Seis são **compostas**, precisam de duas informações, às vezes de dois documentos. Seis **não têm resposta** no manual, e a resposta certa para elas é dizer que o manual não cobre. Cada pergunta traz o documento onde a resposta está (`gold_docs`), uma resposta de referência e uma lista de palavras que a resposta certa precisa conter.

### 🔮 Preveja
Qual é o documento mais longo do corpus e quantos tokens ele tem? E se `chunk_size` for 512, quantos chunks o corpus inteiro vai gerar? Escreva os dois números.

### ▶️ Rode

In [ ]:
docs = load_corpus()
questions = json.load(open("questions.json", encoding="utf-8"))

print(f"{'documento':<28} {'tokens':>6}")
for d in sorted(docs, key=lambda d: -count_tokens(d["text"])):
    print(f"{d['doc']:<28} {count_tokens(d['text']):>6}")
print(f"{'total':<28} {sum(count_tokens(d['text']) for d in docs):>6}")

print()
for tier in ("direct", "multi", "none"):
    qs = [q for q in questions if q["tier"] == tier]
    print(f"{tier:<7} {len(qs)} perguntas · ex.: {qs[0]['question']}")

### 🔍 O que você deve ver

`handbook_procedures.md` é o maior, com cerca de 950 tokens (palavras). O corpus inteiro tem uns 2.800. Isso é pequeno de propósito. Pequeno o suficiente para você ler tudo em dez minutos e saber, para cada pergunta, onde a resposta está. Um RAG de verdade tem milhares de documentos, mas o método de avaliação é exatamente este, só que com mais linhas.

Repare nas três faixas. As perguntas sem resposta não são pegadinha. Elas medem a coisa mais importante de um assistente aterrado em documentos: saber calar.

### 🧩 Preencha a lacuna
Escreva `tokens_por_doc(docs)` que devolve um dicionário `{nome: tokens}` e use-o para responder: quantos documentos têm menos de 200 tokens?

In [ ]:
def tokens_por_doc(docs) -> dict:
    # ---- SEU CÓDIGO AQUI ----
    ...

t = tokens_por_doc(docs)
print(sum(1 for v in t.values() if v < 200), "documentos com menos de 200 tokens")

🧪 **Como saber se deu certo.** O número deve ser 7.

<details>
<summary><b>🔑 Solução de referência</b> (abra só depois dos dez minutos)</summary>

```python
def tokens_por_doc(docs) -> dict:
    return {d["doc"]: count_tokens(d["text"]) for d in docs}
```
</details>

### 🔒 Fechamento
*"Corpus, perguntas e referências ficam fixos antes de qualquer código de RAG. É contra eles que tudo será medido."*

### 🧭 Por que o experimento é assim
O corpus é fictício para que o ganho do RAG seja inequívoco. Se usássemos a Wikipédia, o modelo poderia acertar de memória e você não saberia se a recuperação funcionou. Com o carro 27, toda resposta certa veio dos documentos.

### 🐇 Toca do coelho
O `radio_messages.csv` da aula 7 são 30 mensagens de rádio da mesma equipe. Transforme cada mensagem num documento e adicione ao corpus. Que perguntas novas passam a ter resposta?

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 0

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 1 · Chunking

### 🎯 Uma ideia
**`chunk_size` decide o que viaja junto. Pequeno demais separa o fato da sua condição; grande demais enterra o fato entre outros.**

`chunk_fixed` corta o texto a cada `chunk_size` tokens e repete os últimos `overlap` tokens no início do pedaço seguinte, para uma ideia que atravessa a costura não ser cortada ao meio. É a estratégia mais burra e a mais previsível, e é a nossa linha de base. A aula 9 troca o cortador e mede se melhorou.

### 🔮 Preveja
Vamos fatiar `handbook_procedures.md` com 128, 512 e 2048 tokens e olhar o chunk que contém "23.5 psi". Que fração da seção 4.2 (o procedimento molhado inteiro) você acha que viaja junto em cada tamanho? E quantas outras seções entram no mesmo chunk?

### ▶️ Rode

In [ ]:
page = next(d for d in docs if d["doc"] == "handbook_procedures.md")
print(f"handbook_procedures.md tem {count_tokens(page['text'])} tokens\n")

# a seção 4.2 inteira, para medir quanto dela viaja junto com a linha da pressão
sec42 = next(s for s in page["text"].split("## ") if s.startswith("4.2"))
sentences42 = [x.strip() for x in re.split(r"(?<=[.!?])\s+", sec42) if len(x.strip()) > 20]

def coverage(chunk):                       # fração das frases da seção 4.2 presentes no chunk
    return sum(1 for x in sentences42 if x in chunk) / len(sentences42)

for size in (128, 512, 2048):
    chunks = chunk_fixed(page["text"], chunk_size=size, overlap=size // 10)
    hit = next(i for i, c in enumerate(chunks) if "23.5 psi" in c)
    c = chunks[hit]
    print(f"chunk_size={size:<5} {len(chunks)} chunks · a pressão está no chunk {hit} ({count_tokens(c)} tokens)")
    print(f"    quanto da seção 4.2 (wet) veio junto: {coverage(c):.0%}")
    print(f"    outras seções no mesmo chunk: {[x for x in re.findall(r'^## (4\.\d)', c, re.M) if x != '4.2'] or 'nenhuma'}\n")

### 🔍 O que você deve ver

Com 128, o chunk que tem a pressão carrega menos da metade da seção 4.2: o valor vem, mas a regra das quatro chamadas de "check" e a nota sobre o aquecimento ficam em outro chunk. Com 512, a seção 4.2 vem inteira (100%), e sobra espaço para o começo das seções vizinhas. Com 2048, o chunk é a página toda, e a pressão divide o mesmo vetor com o procedimento seco, o safety car e a bandeira vermelha.

Isso é o trade-off do slide. Não há tamanho certo universal. Há o tamanho certo para este corpus e para estas perguntas, e o Bloco 5 vai medi-lo.

### 🧩 Preencha a lacuna
Escreva `chunk_paragraphs(text, max_tokens)`, que divide o texto nas quebras de parágrafo (`\n\n`) e junta parágrafos consecutivos enquanto a soma couber em `max_tokens`. Compare a contagem de chunks com `chunk_fixed` para o mesmo limite.

In [ ]:
def chunk_paragraphs(text: str, max_tokens: int = 512) -> list[str]:
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for p in paras:
        # ---- SEU CÓDIGO AQUI ----
        # se count_tokens(current + p) couber em max_tokens, acumule; senão feche o chunk atual e comece outro
        ...
    if current:
        chunks.append(current)
    return chunks

for size in (128, 512):
    a, b = chunk_fixed(page["text"], size, size // 10), chunk_paragraphs(page["text"], size)
    print(f"max {size}: fixo -> {len(a)} chunks · por parágrafo -> {len(b)} chunks · maior chunk por parágrafo: {max(count_tokens(c) for c in b)} tokens")

🧪 **Como saber se deu certo.** Nenhum chunk por parágrafo pode passar de `max_tokens` (a menos que um único parágrafo seja maior que o limite). Com 512, o cortador por parágrafo deve produzir 2 ou 3 chunks, e nenhum deles corta uma frase ao meio.

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def chunk_paragraphs(text: str, max_tokens: int = 512) -> list[str]:
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for p in paras:
        candidate = (current + "\n\n" + p).strip() if current else p
        if count_tokens(candidate) <= max_tokens:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = p
    if current:
        chunks.append(current)
    return chunks
```
</details>

### 🔒 Fechamento
*"O chunk é a unidade de recuperação. O que não está no mesmo chunk não chega junto ao modelo."*

### 🧭 Por que o experimento é assim
Usamos palavras como tokens para o notebook não depender de um tokenizador externo. Um tokenizador BPE real gera cerca de 30% mais tokens que palavras, então `chunk_size=512` aqui equivale a uns 650 tokens de modelo. A proporção entre os tamanhos, que é o que a aula discute, não muda.

### 🐇 Toca do coelho
Instale `tiktoken` e troque `tokenize`/`detokenize` por `enc.encode`/`enc.decode` com `cl100k_base`. Quantos tokens tem o corpus agora? A resposta do Bloco 5 muda?

In [ ]:
# 🐇 Espaço livre para a toca do coelho.

### 📓 Diário de bordo · Bloco 1

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 2 · Embeddings

### 🎯 Uma ideia
**Um embedding de chunk é um vetor treinado para que textos com o mesmo significado fiquem perto, mesmo sem nenhuma palavra em comum. A proximidade é o cosseno.**

Na aula 6 cada token do GPT-2 tinha um vetor de 768 dimensões feito para prever o próximo token. Aqui o `nomic-embed-text` devolve um vetor por texto, com 768 dimensões, feito para comparar. Mesma ideia de coordenadas, objetivo de treino diferente.

### 🔮 Preveja
Vamos calcular o cosseno entre a pergunta *"rain procedure for the pit stop"* e três frases: a do procedimento molhado, a da biografia da piloto do 88 e a de compostos de pneu. Ordene as três antes de rodar. Repare que a pergunta não contém "wet" nem "pressure".

### ▶️ Rode

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = {
    "A · wet procedure": "For car 27 in wet conditions the front tyres are set to 23.5 psi and the rear tyres to 21.0 psi.",
    "B · driver bio":    "Mika Sørensen drives car 88. She was born in Aarhus, Denmark, in 2001.",
    "C · compounds":     "Five dry compounds are available to the team during a season, named C1 to C5.",
}
query = "rain procedure for the pit stop"

vecs = dict(zip(sentences, embed(list(sentences.values()))))
q = embed([query])[0]
print(f"dimensões do vetor: {len(q)}\n")
for name, v in sorted(vecs.items(), key=lambda kv: -cosine(q, kv[1])):
    print(f"cos(query, {name}) = {cosine(q, v):.3f}")

### 🔍 O que você deve ver

A frase A vence com folga, mesmo sem compartilhar "rain" com a pergunta. Isso é a busca por vizinhança de significado do slide da galáxia. B e C ficam bem abaixo. Os valores absolutos variam com o modelo de embedding (com o `nomic-embed-text` tudo tende a ficar entre 0.3 e 0.9, porque o espaço é "apertado"); o que importa é a ordem e a distância entre a primeira e a segunda.

Agora a galáxia inteira. A célula abaixo embute todos os chunks de 512 tokens, projeta os 768 números em 2 com PCA e desenha. É uma sombra achatada da galáxia, mas as constelações aparecem.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

rows = make_chunks(docs, chunk_size=512, overlap=51)
X = np.asarray(embed([r["text"] for r in rows]))
qv = np.asarray(embed(["What tyre pressure do we use for car 27 in the wet?"]))

P = PCA(n_components=2).fit(np.vstack([X, qv]))
X2, q2 = P.transform(X), P.transform(qv)

plt.figure(figsize=(8, 5))
plt.scatter(X2[:, 0], X2[:, 1], s=60, c="#1F3161")
for (x, y), r in zip(X2, rows):
    plt.annotate(r["doc"].replace(".md", "") + f"#{r['chunk']}", (x, y), fontsize=7, xytext=(4, 3), textcoords="offset points")
plt.scatter(q2[:, 0], q2[:, 1], s=140, c="#6E9278", marker="*", label="the question")
plt.legend(); plt.title("Chunks do manual em 2D (PCA de 768 dimensões)"); plt.xticks([]); plt.yticks([]); plt.show()

### 🔍 O que você deve ver

A estrela verde (a pergunta) cai perto dos chunks de `handbook_procedures` e `handbook_tyres`, e longe de `logistics` e `sponsor_obligations`. A projeção em 2D perde muito (768 dimensões viraram 2), então vizinhos no gráfico nem sempre são vizinhos de verdade. Use-a como intuição, não como medida.

### 🧩 Preencha a lacuna
Implemente `nearest(query_vec, rows, X, k)` que devolve os `k` chunks mais próximos pela função `cosine`, sem usar o Chroma. É a busca do Bloco 3 feita à mão.

In [ ]:
def nearest(query_vec, rows, X, k=3):
    # ---- SEU CÓDIGO AQUI ----
    # calcule cosine(query_vec, X[i]) para cada i, ordene do maior para o menor e devolva os k primeiros (row, score)
    ...

for r, score in nearest(qv[0], rows, X, k=3):
    print(f"{score:.3f}  {r['doc']} · {r['section']}")

🧪 **Como saber se deu certo.** O primeiro resultado deve ser um chunk de `handbook_procedures.md` (seção 4.2) ou de `handbook_tyres.md` (tabela de pressões), com cosseno acima de 0.6.

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def nearest(query_vec, rows, X, k=3):
    scores = [(r, cosine(query_vec, x)) for r, x in zip(rows, X)]
    return sorted(scores, key=lambda t: -t[1])[:k]
```
</details>

### 🔒 Fechamento
*"Indexação e consulta usam o mesmo modelo de embedding. Se não usarem, o cosseno não significa nada."*

### 🧭 Por que o experimento é assim
O cosseno à mão em 3 dimensões apareceu no slide; aqui são 768 e a conta é idêntica. Fazer a busca sem o Chroma uma vez desmistifica o banco de vetores: ele é essa mesma ordenação, só que com um índice que evita comparar com todos.

### 🐇 Toca do coelho
Troque o `nomic-embed-text` pelo `mxbai-embed-large` (`ollama pull mxbai-embed-large`). A ordem das três frases muda? E os valores absolutos? Lembre de reconstruir o índice do Bloco 3 com `rebuild=True`, ou o `assert` de `retrieve` vai reclamar.

In [ ]:
# 🐇 Espaço livre para a toca do coelho.

### 📓 Diário de bordo · Bloco 2

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 3 · Vector store e retrieval

### 🎯 Uma ideia
**O banco de vetores guarda três coisas por chunk, texto, vetor e metadado, e responde a uma pergunta só: quais são os k vetores mais próximos deste?**

Usamos o Chroma persistente, com distância cosseno (`distance = 1 - cos`). O nome do modelo de embedding fica gravado nos metadados da coleção, e `retrieve` confere esse nome antes de consultar. Esse `assert` é a versão em código do slide "non-negotiable".

### 🔮 Preveja
Para a pergunta do carro 27 com `k=12`, a partir de qual posição da lista você acha que os resultados param de falar de pneus e pit stop?

### ▶️ Rode

In [ ]:
index = build_index(rows, "handbook_512")      # rows são os chunks de 512 do Bloco 2
print(index.count(), "vetores ·", index.metadata)

Q = "What tyre pressure do we use for car 27 in the wet?"
for k in (1, 4, 12):
    print(f"\n===== top_k = {k} =====")
    show(retrieve(index, Q, k=k))

### 🔍 O que você deve ver

Com `k=1`, o chunk da seção 4.2 ou a tabela de pressões. Com `k=4`, os dois e mais dois vizinhos ainda sobre pneus. Com `k=12`, a partir da quinta ou sexta posição entram biografias, logística, patrocínio, e a distância sobe. Tudo isso iria para o prompt. Com um modelo de 3B, o que entra a mais não é neutro: ele lê tudo com a mesma atenção e a resposta piora.

Agora a métrica. Para cada pergunta com resposta, sabemos em que documento ela está (`gold_docs`). **recall@k** é a fração de perguntas em que pelo menos um chunk do documento certo apareceu entre os k primeiros. É a métrica de recuperação, e ela é julgada antes da geração.

### 🧩 Preencha a lacuna
Escreva `recall_at_k(index, questions, k)` e rode para k em 1, 3, 5 e 10 sobre as 14 perguntas que têm resposta.

In [ ]:
answerable = [q for q in questions if not q["expect_refusal"]]

def recall_at_k(index, qs, k) -> float:
    hits_ok = 0
    for q in qs:
        hits = retrieve(index, q["question"], k=k)
        # ---- SEU CÓDIGO AQUI ----
        # conte 1 se algum h["doc"] em hits estiver em q["gold_docs"]
        ...
    return hits_ok / len(qs)

for k in (1, 3, 5, 10):
    print(f"recall@{k:<2} = {recall_at_k(index, answerable, k):.2f}")

🧪 **Como saber se deu certo.** O recall não pode diminuir quando k cresce. Com este corpus e 512 tokens, espere algo em torno de 0.7 a 0.9 em `k=1` e perto de 1.0 em `k=5`. Se `recall@1` ficar abaixo de 0.5, confira se o índice foi construído com o mesmo `EMB_MODEL` da consulta.

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def recall_at_k(index, qs, k) -> float:
    hits_ok = 0
    for q in qs:
        hits = retrieve(index, q["question"], k=k)
        if any(h["doc"] in q["gold_docs"] for h in hits):
            hits_ok += 1
    return hits_ok / len(qs)
```
</details>

### 🔒 Fechamento
*"Se o chunk certo não chega, nenhum prompt salva a resposta. Meço a busca antes de medir a geração."*

### 🧭 Por que o experimento é assim
O recall é calculado por documento, não por chunk, porque é o que dá para rotular à mão em dez minutos. Em produção rotula-se o chunk exato (ou a frase), e a métrica ganha precisão. A ideia é a mesma.

### 🐇 Toca do coelho
Construa um segundo índice com `chunk_size=128` (`build_index(make_chunks(docs, 128, 12), "handbook_128")`) e refaça o `recall@k`. Qual tamanho ganha em `k=1`? E em `k=5`? Guarde a resposta para o exercício.

In [ ]:
# 🐇 Espaço livre para a toca do coelho.

### 📓 Diário de bordo · Bloco 3

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 4 · Geração

### 🎯 Uma ideia
**O prompt aumentado tem três partes na ordem: instrução, evidência, pergunta. O trabalho do modelo deixa de ser lembrar e passa a ser ler, selecionar e escrever.**

`build_prompt` monta exatamente isso. A instrução (`GROUNDING`) manda responder só com o contexto, dizer uma frase fixa quando o contexto não cobre, e citar a fonte entre colchetes. A frase fixa importa: é o que o harness do Bloco 5 procura para saber se o modelo recusou.

### 🔮 Preveja
Vamos fazer a pergunta dos pontos do campeonato, que não está no manual, com e sem a regra de grounding. Escreva a primeira frase de cada resposta.

### ▶️ Rode

In [ ]:
for q in ["What tyre pressure do we use for car 27 in the wet?",
          "How many championship points does car 27 have this season?"]:
    hits = retrieve(index, q, k=4)
    for grounded in (True, False):
        answer, secs = generate(build_prompt(q, hits, grounded=grounded))
        print(f"Q: {q}\n{'COM' if grounded else 'SEM'} grounding ({secs:.1f} s):\n{answer}\n")
    print("-" * 80)

### 🔍 O que você deve ver

Para a primeira pergunta, as duas versões acertam os números, mas só a versão com grounding cita `[handbook_procedures.md]` ou `[handbook_tyres.md]`. Para a segunda, a versão com grounding responde com a frase fixa ("The handbook does not cover this.") e nenhum número. A versão sem grounding inventa uma pontuação e uma posição. É o slide 3 de novo, agora com o interruptor na sua mão.

Repare também que a pergunta dos pontos recuperou quatro chunks mesmo assim (o banco sempre devolve k vizinhos, relevantes ou não). O modelo é que teve de decidir que nenhum deles servia. Essa decisão é o que o CRAG da aula 9 tira do gerador e coloca num avaliador antes dele.

### 🧩 Preencha a lacuna
Escreva `cited_sources(answer)` que extrai os nomes de arquivo citados entre colchetes na resposta, e `citations_are_valid(answer, hits)` que devolve `True` se todas as citações apontam para documentos que estavam no contexto. Um modelo pequeno às vezes cita um arquivo que não recebeu.

In [ ]:
def cited_sources(answer: str) -> set[str]:
    # ---- SEU CÓDIGO AQUI ----
    # dica: re.findall(r"\[([^\]]+\.md)\]", answer)
    ...

def citations_are_valid(answer: str, hits) -> bool:
    # ---- SEU CÓDIGO AQUI ----
    ...

q = "Why does car 88 use a higher rear tyre pressure than car 27 in the wet?"
hits = retrieve(index, q, k=4)
answer, _ = generate(build_prompt(q, hits))
print(answer)
print("\ncitações:", cited_sources(answer), "· válidas?", citations_are_valid(answer, hits))

🧪 **Como saber se deu certo.** `cited_sources` deve devolver um conjunto com pelo menos um `.md`. `citations_are_valid` deve ser `True` quando todos estão em `{h["doc"] for h in hits}`. Rode três vezes; se alguma vez vier `False`, você acabou de ver uma alucinação de citação.

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def cited_sources(answer: str) -> set[str]:
    return set(re.findall(r"\[([^\]]+\.md)\]", answer))

def citations_are_valid(answer: str, hits) -> bool:
    allowed = {h["doc"] for h in hits}
    return cited_sources(answer) <= allowed
```
</details>

### 🔒 Fechamento
*"A recusa é uma funcionalidade. Eu a testo de propósito com perguntas que sei que não estão no corpus."*

### 🧭 Por que o experimento é assim
A frase de recusa é fixa por um motivo de engenharia: um harness precisa detectar a recusa sem outro LLM. Em produção usa-se também um campo estruturado (`{"answered": false}`), a mesma ideia dos contratos JSON da aula 7.

### 🐇 Toca do coelho
Troque a ordem dos chunks no contexto (`hits[::-1]`) e rode a pergunta composta do carro 88 cinco vezes com cada ordem. A resposta muda? Esse é o efeito "lost in the middle" do slide de limites, e o reranking da aula 9 existe por causa dele.

In [ ]:
# 🐇 Espaço livre para a toca do coelho.

### 📓 Diário de bordo · Bloco 4

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 5 · O harness

### 🎯 Uma ideia
**Um harness roda todas as perguntas, guarda tudo o que aconteceu e cospe uma tabela. Ele é o que permite dizer "melhorou" com número.**

`run_harness` recebe uma função de recuperação e um `k`, roda as 20 perguntas com e sem RAG, e devolve um DataFrame com o chunk recuperado, a resposta, se acertou, se recusou quando devia e a latência. A nota é simples de propósito: uma resposta está certa se contém todas as palavras de `must_contain`; uma recusa está certa se contém a frase fixa. É grosseiro e é auditável, e você pode ler as 20 linhas em cinco minutos.

### 🔮 Preveja
Sem RAG, quantas das 8 perguntas diretas o `qwen2.5:3b` vai acertar? E das 6 sem resposta, em quantas ele vai admitir que não sabe?

### ▶️ Rode

In [ ]:
import pandas as pd

REFUSAL = "the handbook does not cover this"

def grade(answer: str, q: dict) -> tuple[bool, bool]:
    """(acertou, recusou). Acerto = todas as palavras-chave presentes, ou recusa quando era esperada."""
    a = answer.lower()
    refused = REFUSAL in a
    if q["expect_refusal"]:
        return refused, refused
    return all(kw.lower() in a for kw in q["must_contain"]) and not refused, refused

def run_harness(index, questions, k=4, retriever=retrieve, model=GEN_MODEL, label="baseline"):
    records = []
    for q in questions:
        for mode in ("no_rag", "rag"):
            hits = retriever(index, q["question"], k=k) if mode == "rag" else []
            answer, secs = generate(build_prompt(q["question"], hits, grounded=(mode == "rag")), model=model)
            correct, refused = grade(answer, q)
            records.append({"label": label, "model": model, "k": k, "mode": mode, "id": q["id"], "tier": q["tier"],
                            "question": q["question"], "retrieved_docs": "|".join(h["doc"] for h in hits),
                            "gold_hit": any(h["doc"] in q["gold_docs"] for h in hits),
                            "answer": answer, "correct": correct, "refused": refused, "latency_s": round(secs, 2)})
            print(f"{q['id']} {mode:<6} {'✓' if correct else '✗'}  {secs:4.1f}s")
    return pd.DataFrame(records)

df = run_harness(index, questions, k=4)
df.to_csv("results_baseline.csv", index=False)
df.groupby(["mode", "tier"])["correct"].mean().unstack().round(2)

### 🔍 O que você deve ver

Uma tabela com duas linhas (`no_rag`, `rag`) e três colunas (`direct`, `multi`, `none`). Sem RAG, as diretas ficam perto de zero (o modelo não tem como saber) e as sem resposta também ficam baixas (ele inventa em vez de recusar). Com RAG, as diretas sobem para perto de 1.0, as compostas ficam no meio (às vezes o segundo fato não veio, às vezes veio e o modelo não juntou) e as sem resposta sobem porque a regra de grounding funciona.

Leia as linhas erradas de `df[df.correct == False]`. Cada uma cai numa de três caixas: o chunk certo não veio (`gold_hit == False`, problema de recuperação), o chunk veio e o modelo errou (problema de geração), ou a nota grosseira errou (o modelo respondeu certo com outras palavras). Separar essas três caixas é o trabalho de quem constrói RAG.

### 🧩 Preencha a lacuna
Escreva `sweep(index_by_size, ks)` que roda o harness para cada índice em `index_by_size` (um dicionário `{chunk_size: index}`) e cada `k` em `ks`, só no modo `rag`, e devolve uma tabela `chunk_size × k` com o acerto médio. Comece com dois tamanhos e dois valores de k para não esperar demais.

In [ ]:
index_by_size = {
    512: index,
    128: build_index(make_chunks(docs, 128, 12), "handbook_128"),
}

def sweep(index_by_size: dict, ks: list[int]) -> pd.DataFrame:
    rows_out = []
    for size, idx in index_by_size.items():
        for k in ks:
            # ---- SEU CÓDIGO AQUI ----
            # rode run_harness(idx, questions, k=k, label=f"cs{size}_k{k}"), filtre mode == "rag",
            # e acrescente {"chunk_size": size, "k": k, "accuracy": média de correct}
            ...
    return pd.DataFrame(rows_out).pivot(index="chunk_size", columns="k", values="accuracy")

sweep(index_by_size, ks=[1, 4])

🧪 **Como saber se deu certo.** Uma tabela 2 × 2. O que esperar: com `k=1`, 512 costuma vencer 128 (o fato e a condição viajam juntos); com `k=4`, a diferença diminui porque os chunks de 128 se completam. Se os quatro números forem iguais, verifique se `run_harness` está mesmo usando o índice de cada tamanho.

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def sweep(index_by_size: dict, ks: list[int]) -> pd.DataFrame:
    rows_out = []
    for size, idx in index_by_size.items():
        for k in ks:
            d = run_harness(idx, questions, k=k, label=f"cs{size}_k{k}")
            acc = d[d["mode"] == "rag"]["correct"].mean()
            rows_out.append({"chunk_size": size, "k": k, "accuracy": round(acc, 2)})
    return pd.DataFrame(rows_out).pivot(index="chunk_size", columns="k", values="accuracy")
```
</details>

### 🔒 Fechamento
*"Mesmo corpus, mesmas perguntas, mesmo harness. Qualquer mudança no pipeline vira uma linha nova na mesma tabela."*

### 🧭 Por que o experimento é assim
O harness recebe `retriever` como parâmetro por um motivo só: na aula 9, `retrieve` será substituída por `retrieve_hybrid`, `retrieve_hyde`, `retrieve_crag`, e o resto do código não muda. O `results_baseline.csv` é a linha contra a qual todas elas serão comparadas.

### 🐇 Toca do coelho
Troque a nota por palavras-chave por um juiz LLM: um prompt que recebe pergunta, referência e resposta e devolve `correct: true/false` em JSON (aula 7, Bloco 5). Em quantas linhas o juiz discorda da nota grosseira? Quem tem razão?

In [ ]:
# 🐇 Espaço livre para a toca do coelho.

### 📓 Diário de bordo · Bloco 5

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# 📝 Auto-teste final (recuperação espaçada)

Responda **sem rolar o notebook**, de preferência um ou dois dias depois. Cada pergunta corresponde a um 🔒.

1. Quais são as três coisas que ficam fixas antes de qualquer código de RAG, e por quê?
2. O que muda no que "viaja junto" quando `chunk_size` passa de 128 para 2048?
3. Por que indexação e consulta precisam usar o mesmo modelo de embedding? O que acontece se não usarem?
4. O que `recall@k` mede e por que ele é julgado antes da geração?
5. Quais são as três partes do prompt aumentado, na ordem? Qual é o papel da frase fixa de recusa?
6. Um RAG errou uma pergunta. Quais são as três caixas em que o erro pode cair, e que coluna do harness separa a primeira das outras duas?
7. Por que o harness recebe a função de recuperação como parâmetro?

---
# 🏁 Exercício da semana · A linha de base

Na aula 9 cada variante sofisticada do RAG vai rodar sobre este mesmo corpus e ser comparada com o que você produzir agora. O exercício é produzir uma linha de base que valha a pena bater.

> **Qual combinação de `chunk_size`, `top_k` e modelo você colocaria em produção para o manual da Aurora, e por quê?**

### Construir

1. Rode o harness completo (modos `no_rag` e `rag`) para dois modelos, `qwen2.5:3b` e `qwen2.5:0.5b`, com `chunk_size=512` e `k=4`. Guarde os dois CSVs.
2. Rode a varredura do Bloco 5 com `chunk_size` em {128, 256, 512, 1024} e `k` em {1, 3, 5}, só com o `qwen2.5:3b`, só no modo `rag`. Guarde o CSV concatenado.
3. Para a melhor combinação, calcule também `recall@k` e a latência média.

### Analisar

4. Leia todas as linhas erradas da melhor combinação e classifique cada uma nas três caixas (recuperação, geração, nota). Uma tabela com `id`, caixa e uma frase.
5. Um parágrafo sobre as perguntas sem resposta: em quantas o modelo recusou, em quantas inventou, e o que mudou entre os dois modelos.

### Entregar

Um relatório de **duas páginas** (PDF ou notebook exportado) com as tabelas dos itens 1 a 3, a tabela do item 4, o parágrafo do item 5 e a sua recomendação final em três linhas. Anexe os CSVs. **Não apague `results_baseline.csv`.** Ele é a primeira linha da tabela da aula 9.

### Bônus (opcional)

Implemente `chunk_paragraphs` como estratégia do índice (o `make_chunks` aceita um cortador diferente com uma mudança de duas linhas) e adicione uma linha à tabela do item 2. É a primeira "variante" da aula 9, feita por você antes da aula.

---
# ⏱ Tempos das células

Um retrato de quanto custou cada passo nesta máquina. Compare a indexação (embedding de 13 chunks) com uma única resposta do gerador, e o harness com tudo o mais. Anexe esta tabela ao relatório do exercício.

In [ ]:
# Resumo dos tempos de todas as células rodadas nesta sessão.
import pandas as pd
times = pd.DataFrame(CELL_TIMES)
print(f"total: {times['seconds'].sum():.1f} s em {len(times)} células")
times


---
# 🗺️ Onde tudo se conecta

| Você viu | Em | Volta como |
|---|---|---|
| Embeddings de token no GPT-2 | Aula 06 | Embeddings de chunk, Bloco 2 |
| O Ollama como servidor HTTP | Aula 07 | `/api/embed` e `/v1/chat`, todo o notebook |
| Temperatura e `seed` | Aula 07 | `temperature=0` no gerador, para o harness ser repetível |
| Contrato JSON entre modelo e software | Aula 07, Blocos 5 e 6 | A frase fixa de recusa e a toca do coelho do juiz LLM |
| Os carros 27 e 88 | Aula 07 | O corpus inteiro |
| O harness com `retriever` como parâmetro | Bloco 5 | Toda variante da aula 09 |
| As linhas erradas em três caixas | Bloco 5 | CRAG (recuperação), reranking (geração), juiz LLM (nota), Aula 09 |